# 02 — Clean the raw data

Turns the three raw files into tidy, analysis-ready tables in `data/processed/`. Decisions here follow the takeaways from `01_explore.ipynb`.

Outputs written:

| File | Grain | Notes |
|---|---|---|
| `rtb_rent.csv` | county × quarter × bedrooms × property type | county-level only, rent as number, null rents dropped |
| `earnings.csv` | NUTS 3 region × year | Both sexes; mean & median kept |
| `ppr_sales.csv` | one row per sale | market-rate only, price as number |
| `county_region.csv` | county → NUTS 3 region | lookup for joining rents/sales to earnings |

Nothing is aggregated yet — that happens in the SQL layer.

In [1]:
import pandas as pd
from pathlib import Path

RAW = Path("..") / "data" / "raw"
PROCESSED = Path("..") / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

COUNTIES = [
    "Carlow", "Cavan", "Clare", "Cork", "Donegal", "Dublin", "Galway",
    "Kerry", "Kildare", "Kilkenny", "Laois", "Leitrim", "Limerick",
    "Longford", "Louth", "Mayo", "Meath", "Monaghan", "Offaly",
    "Roscommon", "Sligo", "Tipperary", "Waterford", "Westmeath",
    "Wexford", "Wicklow",
]

## County → NUTS 3 region lookup

CSO publishes earnings only at NUTS 3 region level, so every county is mapped to its parent region (2016 NUTS revision, the version the CSO file uses). This lookup is what lets us divide county rents by regional earnings later.

In [2]:
region_to_counties = {
    "Border":     ["Cavan", "Donegal", "Leitrim", "Monaghan", "Sligo"],
    "West":       ["Galway", "Mayo", "Roscommon"],
    "Mid-West":   ["Clare", "Limerick", "Tipperary"],
    "South-East": ["Carlow", "Kilkenny", "Waterford", "Wexford"],
    "South-West": ["Cork", "Kerry"],
    "Dublin":     ["Dublin"],
    "Mid-East":   ["Kildare", "Louth", "Meath", "Wicklow"],
    "Midland":    ["Laois", "Longford", "Offaly", "Westmeath"],
}

county_region = pd.DataFrame(
    [(county, region) for region, counties in region_to_counties.items() for county in counties],
    columns=["county", "region"],
).sort_values("county", ignore_index=True)

# Sanity: all 26 counties covered exactly once
assert len(county_region) == 26
assert set(county_region["county"]) == set(COUNTIES)
county_region

,county,region
0,Carlow,South-East
1,Cavan,Border
2,Clare,Mid-West
3,Cork,South-West
4,Donegal,Border
5,Dublin,Dublin
6,Galway,West
7,Kerry,South-West
8,Kildare,Mid-East
9,Kilkenny,South-East


## 1. RTB rent index

Keep county-level rows only (drop the town/postal-district detail), parse the quarter, rename to snake_case, coerce rent to a number, and drop rows with no published rent.

In [3]:
rtb_raw = pd.read_csv(RAW / "rtb_rent_index.csv")

rtb = (
    rtb_raw
    .rename(columns={
        "Quarter": "quarter",
        "Number of Bedrooms": "bedrooms",
        "Property Type": "property_type",
        "Location": "county",
        "VALUE": "rent",
    })
    .loc[lambda d: d["county"].isin(COUNTIES)]
    .dropna(subset=["rent"])
    .copy()
)

# 2014Q1 -> period and a sortable quarter-start date
rtb["period"] = pd.PeriodIndex(rtb["quarter"].str.replace("Q", "-Q"), freq="Q")
rtb["date"] = rtb["period"].dt.start_time
rtb["year"] = rtb["period"].dt.year
rtb["rent"] = rtb["rent"].astype(float)

rtb = rtb[["county", "quarter", "date", "year", "bedrooms", "property_type", "rent"]]
print("rows:", len(rtb), "| counties:", rtb["county"].nunique(), "| quarters:", rtb["quarter"].nunique())
print("rent range:", round(rtb['rent'].min()), "to", round(rtb['rent'].max()))
rtb.head()

rows: 40201 | counties: 26 | quarters: 47
rent range: 260 to 3782


,county,quarter,date,year,bedrooms,property_type,rent
0,Carlow,2014Q1,2014-01-01,2014,All bedrooms,All property types,579.65
4,Cavan,2014Q1,2014-01-01,2014,All bedrooms,All property types,441.91
13,Clare,2014Q1,2014-01-01,2014,All bedrooms,All property types,531.53
20,Cork,2014Q1,2014-01-01,2014,All bedrooms,All property types,733.50
62,Donegal,2014Q1,2014-01-01,2014,All bedrooms,All property types,440.03


## 2. CSO earnings

Filter to "Both sexes", pivot mean and median into their own columns, one row per region-year. The README notes earnings stop at 2024.

In [4]:
earn_raw = pd.read_csv(RAW / "cso_earnings.csv")

earn = earn_raw[earn_raw["Sex"] == "Both sexes"].copy()
earn["measure"] = earn["Statistic Label"].map({
    "Mean Annual Earnings": "mean_earnings",
    "Median Annual Earnings": "median_earnings",
})

earnings = (
    earn.pivot_table(index=["NUTS 3 Regions", "Year"], columns="measure", values="VALUE")
    .reset_index()
    .rename(columns={"NUTS 3 Regions": "region", "Year": "year"})
    .rename_axis(columns=None)
)

print("rows:", len(earnings), "| years:", earnings['year'].min(), "to", earnings['year'].max())
print("regions:", sorted(earnings['region'].unique()))
earnings.head()

rows: 126 | years: 2011 to 2024
regions: ['Border', 'Dublin', 'Mid-East', 'Mid-West', 'Midland', 'South-East', 'South-West', 'State', 'West']


,region,year,mean_earnings,median_earnings
0,Border,2011,33572.0,29380.0
1,Border,2012,33372.0,29146.0
2,Border,2013,33452.0,29194.0
3,Border,2014,33579.0,29431.0
4,Border,2015,34011.0,29777.0


The earnings file labels the national total "State". The eight analysis regions match the lookup; "State" is kept in the file as a handy national benchmark but won't join to any county.

## 3. Property Price Register

Parse the `€`-prefixed price string to a float, parse the date, keep market-rate sales only (`Not Full Market Price == "No"`), and trim to the columns the analysis needs. Read with `low_memory=False` to avoid the mixed-type warning seen in exploration.

In [5]:
ppr_raw = pd.read_csv(RAW / "ppr.csv", encoding="cp1252", low_memory=False)
price_col = next(c for c in ppr_raw.columns if c.startswith("Price"))

ppr = ppr_raw.rename(columns={
    "Date of Sale (dd/mm/yyyy)": "date",
    "County": "county",
    price_col: "price",
    "Not Full Market Price": "not_full_market_price",
    "Description of Property": "description",
}).copy()

ppr["date"] = pd.to_datetime(ppr["date"], format="%d/%m/%Y")
ppr["year"] = ppr["date"].dt.year
# strip euro sign, thousands commas, stray spaces -> float
ppr["price"] = (
    ppr["price"].astype(str)
    .str.replace(r"[^0-9.]", "", regex=True)
    .astype(float)
)

before = len(ppr)
ppr = ppr[ppr["not_full_market_price"] == "No"]
print(f"market-rate filter: kept {len(ppr):,} of {before:,} ({len(ppr)/before:.1%})")

ppr = ppr[["date", "year", "county", "price", "description"]].reset_index(drop=True)
print("price range:", f"EUR {ppr['price'].min():,.0f}", "to", f"EUR {ppr['price'].max():,.0f}")
print("median price:", f"EUR {ppr['price'].median():,.0f}")
ppr.head()

market-rate filter: kept 742,711 of 782,596 (94.9%)
price range: EUR 5,031 to EUR 387,665,198
median price: EUR 246,000


,date,year,county,price,description
0,2010-01-01,2010,Dublin,343000.0,Second-Hand Dwelling house /Apartment
1,2010-01-03,2010,Laois,185000.0,New Dwelling house /Apartment
2,2010-01-04,2010,Dublin,438500.0,Second-Hand Dwelling house /Apartment
3,2010-01-04,2010,Meath,400000.0,Second-Hand Dwelling house /Apartment
4,2010-01-04,2010,Kilkenny,160000.0,Second-Hand Dwelling house /Apartment


## Write processed files

In [6]:
rtb.to_csv(PROCESSED / "rtb_rent.csv", index=False)
earnings.to_csv(PROCESSED / "earnings.csv", index=False)
ppr.to_csv(PROCESSED / "ppr_sales.csv", index=False)
county_region.to_csv(PROCESSED / "county_region.csv", index=False)

for name, df in [
    ("rtb_rent.csv", rtb),
    ("earnings.csv", earnings),
    ("ppr_sales.csv", ppr),
    ("county_region.csv", county_region),
]:
    print(f"{name:20s} {len(df):>10,} rows  ->  {', '.join(df.columns)}")

rtb_rent.csv             40,201 rows  ->  county, quarter, date, year, bedrooms, property_type, rent
earnings.csv                126 rows  ->  region, year, mean_earnings, median_earnings
ppr_sales.csv           742,711 rows  ->  date, year, county, price, description
county_region.csv            26 rows  ->  county, region
